In [1]:
# full-batch training: double precision

%reset -f
import time
import torch
import torch.nn as nn
import torch.nn.functional as F

# device and precision
device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.set_default_dtype(torch.float64)

# model
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(2, 64), nn.SiLU(),
            nn.Linear(64, 64), nn.SiLU(),
            nn.Linear(64, 64), nn.SiLU(),
            nn.Linear(64, 64), nn.SiLU(),
            nn.Linear(64, 1) )
    def forward(self, x):
        return self.layers(x)
torch.manual_seed(42)
model = Net().to(device)

# training data
N = 200000
torch.manual_seed(99)
xtrain = torch.randn(N, 2, device=device)
ytrain = torch.randn(N, 1, device=device)

# optimizer
optimizer = torch.optim.AdamW(model.parameters())
epochs = 100

# training
torch.cuda.synchronize()
t0 = time.time()
for epoch in range(epochs):
    optimizer.zero_grad()
    pred = model(xtrain)
    loss = F.mse_loss(pred, ytrain)
    loss.backward()
    optimizer.step()
torch.cuda.synchronize()
elapsed = time.time() - t0

print(f'Full-batch training, double precision')
print(f'time for {epochs} epochs = {elapsed:.3f} s')
print(f'time per epoch = {elapsed/epochs:.5f} s')


Full-batch training, double precision
time for 100 epochs = 5.859 s
time per epoch = 0.05859 s


In [2]:
# full-batch training: single precision

%reset -f
import time
import torch
import torch.nn as nn
import torch.nn.functional as F

# device and precision
device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.set_default_dtype(torch.float32)

# model
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(2, 64), nn.SiLU(),
            nn.Linear(64, 64), nn.SiLU(),
            nn.Linear(64, 64), nn.SiLU(),
            nn.Linear(64, 64), nn.SiLU(),
            nn.Linear(64, 1) )
    def forward(self, x):
        return self.layers(x)
torch.manual_seed(42)
model = Net().to(device)

# training data
N = 200000
torch.manual_seed(99)
xtrain = torch.randn(N, 2, device=device)
ytrain = torch.randn(N, 1, device=device)

# optimizer
optimizer = torch.optim.AdamW(model.parameters())
epochs = 100

# training
torch.cuda.synchronize()
t0 = time.time()
for epoch in range(epochs):
    optimizer.zero_grad()
    pred = model(xtrain)
    loss = F.mse_loss(pred, ytrain)
    loss.backward()
    optimizer.step()
torch.cuda.synchronize()
elapsed = time.time() - t0

print(f'Full-batch training, single precision')
print(f'time for {epochs} epochs = {elapsed:.3f} s')
print(f'time per epoch = {elapsed/epochs:.5f} s')


Full-batch training, single precision
time for 100 epochs = 1.051 s
time per epoch = 0.01051 s


In [3]:
# mini-batch training with ** No shuffle **
# dataset on GPU vs CPU: ** DataLoader is faster with dataset on CPU!! **

%reset -f
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

# device and precision
device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.set_default_dtype(torch.float64)

# model
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(2, 64), nn.SiLU(),
            nn.Linear(64, 64), nn.SiLU(),
            nn.Linear(64, 64), nn.SiLU(),
            nn.Linear(64, 64), nn.SiLU(),
            nn.Linear(64, 1) )
    def forward(self, x):
        return self.layers(x)
torch.manual_seed(42)
model = Net().to(device)

# training data
N = 200000
torch.manual_seed(99)
xtrain = torch.randn(N, 2, device=device)
ytrain = torch.randn(N, 1, device=device)

# optimizer
optimizer = torch.optim.AdamW(model.parameters())
epochs = 100

# training
torch.cuda.synchronize()
t0 = time.time()

batch_size = N//20
dataset = TensorDataset(xtrain, ytrain)
loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
for epoch in range(epochs):
    for x, y in  loader:
        optimizer.zero_grad()
        pred = model(x)
        loss = F.mse_loss(pred, y)
        loss.backward()
        optimizer.step()
torch.cuda.synchronize()
elapsed = time.time() - t0

print(f'Mini-batch training using DataLoader with dataset on GPU')
print(f'time for {epochs} epochs = {elapsed:.3f} s')
print(f'time per epoch = {elapsed/epochs:.5f} s')

t0 = time.time()
dataset = TensorDataset(xtrain.cpu(), ytrain.cpu())
loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
for epoch in range(epochs):
    for x, y in  loader:
        optimizer.zero_grad()
        pred = model(x.to(device))
        loss = F.mse_loss(pred, y.to(device))
        loss.backward()
        optimizer.step()
torch.cuda.synchronize()
elapsed = time.time() - t0

print(f'Mini-batch training using DataLoader with dataset on CPU')
print(f'time for {epochs} epochs = {elapsed:.3f} s')
print(f'time per epoch = {elapsed/epochs:.5f} s')


Mini-batch training using DataLoader with dataset on GPU
time for 100 epochs = 214.797 s
time per epoch = 2.14797 s
Mini-batch training using DataLoader with dataset on CPU
time for 100 epochs = 186.883 s
time per epoch = 1.86883 s


In [4]:
# mini-batch training with ** No shuffle ** 
# Just load data, ** No loss computation, No Training **

%reset -f
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

# device and precision
device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.set_default_dtype(torch.float64)

# model
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(2, 64), nn.SiLU(),
            nn.Linear(64, 64), nn.SiLU(),
            nn.Linear(64, 64), nn.SiLU(),
            nn.Linear(64, 64), nn.SiLU(),
            nn.Linear(64, 1) )
    def forward(self, x):
        return self.layers(x)
torch.manual_seed(42)
model = Net().to(device)

# training data
N = 200000
torch.manual_seed(99)
xtrain = torch.randn(N, 2, device=device)
ytrain = torch.randn(N, 1, device=device)

# optimizer
optimizer = torch.optim.AdamW(model.parameters())
epochs = 100

# training
torch.cuda.synchronize()
t0 = time.time()

batch_size = N//20
dataset = TensorDataset(xtrain, ytrain)
loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
for epoch in range(epochs):
    for x, y in  loader:
        pass
torch.cuda.synchronize()
elapsed = time.time() - t0

print(f'Mini-batch training using DataLoader with dataset on GPU')
print(f'    Just load data, ** No loss computation, No Training **')
print(f'time for {epochs} epochs = {elapsed:.3f} s')
print(f'time per epoch = {elapsed/epochs:.5f} s')

t0 = time.time()
dataset = TensorDataset(xtrain.cpu(), ytrain.cpu())
loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
for epoch in range(epochs):
    for x, y in  loader:
        pass
torch.cuda.synchronize()
elapsed = time.time() - t0

print(f'Mini-batch training using DataLoader with dataset on CPU')
print(f'    Just load data, ** No loss computation, No Training **')
print(f'time for {epochs} epochs = {elapsed:.3f} s')
print(f'time per epoch = {elapsed/epochs:.5f} s')


Mini-batch training using DataLoader with dataset on GPU
    Just load data, ** No loss computation, No Training **
time for 100 epochs = 209.718 s
time per epoch = 2.09718 s
Mini-batch training using DataLoader with dataset on CPU
    Just load data, ** No loss computation, No Training **
time for 100 epochs = 178.431 s
time per epoch = 1.78431 s


In [5]:
# mini-batch training: ** manual mini batch **

%reset -f
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

# device and precision
device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.set_default_dtype(torch.float64)

# model
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(2, 64), nn.SiLU(),
            nn.Linear(64, 64), nn.SiLU(),
            nn.Linear(64, 64), nn.SiLU(),
            nn.Linear(64, 64), nn.SiLU(),
            nn.Linear(64, 1) )
    def forward(self, x):
        return self.layers(x)
torch.manual_seed(42)
model = Net().to(device)

# training data
N = 200000
torch.manual_seed(99)
xtrain = torch.randn(N, 2, device=device)
ytrain = torch.randn(N, 1, device=device)

# optimizer
optimizer = torch.optim.AdamW(model.parameters())
epochs = 100

# training
torch.cuda.synchronize()
t0 = time.time()

batch_size = N//20
for epoch in range(epochs):
    perm = torch.randperm(N, device=device)
    xd_perm = xtrain[perm]; yd_perm = ytrain[perm]
    for k in range(0, N, batch_size):
        optimizer.zero_grad()
        pred = model(xd_perm[k:k+batch_size])
        loss = F.mse_loss(pred, yd_perm[k:k+batch_size])
        loss.backward()
        optimizer.step()
torch.cuda.synchronize()
elapsed = time.time() - t0

print(f'Manual mini-batch training ** WITHOUT ** using DataLoader')
print(f'time for {epochs} epochs = {elapsed:.3f} s')
print(f'time per epoch = {elapsed/epochs:.5f} s')


Manual mini-batch training ** WITHOUT ** using DataLoader
time for 100 epochs = 7.775 s
time per epoch = 0.07775 s
